In [31]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset

In [32]:
class ModuloDataset(Dataset):
    def __init__(self, data):
        self.inputs = data[:,:2]
        self.labels = data[:,2:].squeeze(dim=1)

    def __len__(self):
        return len(self.inputs)

    def __getitem__(self, index):
        return self.inputs[index], self.labels[index]

In [33]:
### Create dataset
modulo_number = 113
data = []

for i in range(modulo_number):
    for j in range(modulo_number):
        data.append([i, j, (i + j) % modulo_number])

data = torch.tensor(data)

In [34]:
train_perecentage = 0.3

shuffled_data = data[torch.randperm(data.shape[0])]
train_idx = int(train_perecentage * shuffled_data.shape[0])
train_split = shuffled_data[:train_idx]
test_split = shuffled_data[train_idx:]
print(train_split.shape[0], test_split.shape[0])

train_dataset = ModuloDataset(train_split)
test_dataset = ModuloDataset(test_split)

3830 8939


In [35]:
from torch.nn import Embedding, MultiheadAttention, Linear, Module, ReLU

class MLP(Module):
    def __init__(self, d_model, d_hidden):
        super().__init__()
        self.up_projection = Linear(d_model, d_hidden)
        self.down_projection = Linear(d_hidden, d_model)
        self.relu = ReLU()

    def forward(self, x):
        h1 = self.up_projection(x)
        r1 = self.relu(h1)
        out = self.down_projection(r1)
        return out

class ModuloTransformer(Module):
    def __init__(self, modulo_number, d_embed, n_heads, d_ff):
        super().__init__()
        self.embed = Embedding(modulo_number, d_embed)
        self.pos_embed = Embedding(2, d_embed)      # hard code to 2 bc seq len is always that
        self.mh_attn = MultiheadAttention(d_embed, n_heads, batch_first=True)     # not masked attention, batch is first dim
        self.mlp = MLP(d_embed, d_ff)
        self.unembed = Linear(d_embed, modulo_number)

    def forward(self, x):
        x_embed = self.embed(x)
        pos_embed = self.pos_embed(torch.arange(2, device=x.device))
        x_embed += pos_embed
        x_attn, _ = self.mh_attn(x_embed, x_embed, x_embed)
        x_attn += x_embed               # residual
        x_ff = x_attn + self.mlp(x_attn) # residual
        logits = self.unembed(x_ff)
        # complicated indexing so that it works if there is a batch dim or not
        return logits[..., 1, :]
    
    def predict(self, x):
        logits = self.forward(x)
        probs = torch.softmax(logits, dim=-1)
        # complicated indexing so that it works if there is a batch dim or not
        prediction = torch.argmax(probs, dim=-1)
        return prediction

In [36]:
len(train_dataset), len(test_dataset)

(3830, 8939)

In [37]:
### TRAINING LOOP
from torch.utils.data import DataLoader
from torch.optim import AdamW
import time

# hypers come from paper
BATCH_SIZE = len(train_dataset)
LR = 0.001
GAMMA = 1.0
EPOCHS = 15_000
METHOD = "GD"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

model = ModuloTransformer(modulo_number, 128, 4, 512).to(DEVICE)
optimizer = AdamW(model.parameters(), LR, weight_decay=GAMMA)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

initial_time = time.time()
end_time = None
total_func_calls = 0

def calculate_accuracy(logits,  y):
    # logits will have shape B x M where B is batch size and M is the modulo number
    # y will have shape B
    preds = torch.argmax(logits, dim=-1)
    bools = preds == y
    return bools.float().mean()

log_filename = f"log_{int(time.time())}"
log_file = open(log_filename, "x")
log_file.write(f"Config:\n{BATCH_SIZE=}\n{LR=}\n{GAMMA=}\n{EPOCHS=}\n{METHOD=}\n{DEVICE=}\n{modulo_number=}\n\n")

train_losses = []
test_losses = []
train_accs = []
test_accs = []
for epoch in range(EPOCHS):
    x_train, y_train = next(iter(train_loader))
    x_train, y_train = x_train.to(DEVICE), y_train.to(DEVICE)
    logits = model(x_train)
    train_acc = calculate_accuracy(logits, y_train)
    loss = nn.functional.cross_entropy(logits, y_train)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    # check performance on test data
    with torch.no_grad():
        x_test, y_test = next(iter(test_loader))
        x_test, y_test = x_test.to(DEVICE), y_test.to(DEVICE)
        logits = model(x_test)
        test_acc = calculate_accuracy(logits, y_test)
        if test_acc == 1.0:
            end_time = time.time()
            total_func_calls = epoch
        test_loss = nn.functional.cross_entropy(logits, y_test)

    train_losses.append(loss.item())
    test_losses.append(test_loss.item())
    train_accs.append(train_acc)
    test_accs.append(test_acc)
    
    if epoch % 100 == 0:
        s = f"Epoch {epoch}: Loss -- Train: {loss.item():.4f} Test: {test_loss.item():.4f}    Accuracy -- Train: {train_acc:.4f} Test: {test_acc:.4f}"
        log_file.write(s + '\n')
        if epoch % 1000 == 0:
            print(s)

s = f'Total time: {end_time - initial_time}s    Total func calls {total_func_calls}'
log_file.write(s + '\n')
print(s)
log_file.close()


Epoch 0: Loss -- Train: 5.1533 Test: 4.9851    Accuracy -- Train: 0.0107 Test: 0.0073
Epoch 1000: Loss -- Train: 0.0142 Test: 19.9682    Accuracy -- Train: 1.0000 Test: 0.0018
Epoch 2000: Loss -- Train: 0.1732 Test: 13.1286    Accuracy -- Train: 1.0000 Test: 0.0029
Epoch 3000: Loss -- Train: 0.1961 Test: 12.8994    Accuracy -- Train: 1.0000 Test: 0.0086
Epoch 4000: Loss -- Train: 0.1693 Test: 5.3902    Accuracy -- Train: 1.0000 Test: 0.1201
Epoch 5000: Loss -- Train: 0.0427 Test: 0.1543    Accuracy -- Train: 1.0000 Test: 0.9867
Epoch 6000: Loss -- Train: 0.0224 Test: 0.0816    Accuracy -- Train: 1.0000 Test: 0.9956
Epoch 7000: Loss -- Train: 0.0121 Test: 0.0519    Accuracy -- Train: 1.0000 Test: 0.9956
Epoch 8000: Loss -- Train: 0.0066 Test: 0.0319    Accuracy -- Train: 1.0000 Test: 0.9969
Epoch 9000: Loss -- Train: 0.0038 Test: 0.0188    Accuracy -- Train: 1.0000 Test: 0.9992
Epoch 10000: Loss -- Train: 0.0022 Test: 0.0120    Accuracy -- Train: 1.0000 Test: 0.9995
Epoch 11000: Loss --

In [41]:
### TRAINING LOOP via LBFGS
from torch.optim import LBFGS

# hypers come from paper
BATCH_SIZE = len(train_dataset)
LR = 1
GAMMA = 0.001
EPOCHS = 100
METHOD = "LBFGS"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

model = ModuloTransformer(modulo_number, 128, 4, 512).to(DEVICE)
model.double()
optimizer = LBFGS(model.parameters(), lr=LR, 
                          tolerance_grad=0, tolerance_change=0, 
                          line_search_fn='strong_wolfe')

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

initial_time = time.time()
end_time = None
total_func_calls = 0

log_filename = f"log_{int(time.time())}"
log_file = open(log_filename, "x")
log_file.write(f"Config:\n{BATCH_SIZE=}\n{LR=}\n{GAMMA=}\n{EPOCHS=}\n{METHOD=}\n{DEVICE=}\n{modulo_number=}\n\n")

train_losses = []
test_losses = []
train_accs = []
test_accs = []
for epoch in range(EPOCHS):
    train_loss_acc = {'loss': None, 'acc': None}

    def closure():
        x_train, y_train = next(iter(train_loader))
        x_train, y_train = x_train.to(DEVICE), y_train.to(DEVICE)
        logits = model(x_train)
        train_acc = calculate_accuracy(logits, y_train)
        loss = nn.functional.cross_entropy(logits, y_train)
        weight_decay = sum(p.pow(2).sum() for p in model.parameters()) # have to manually add regularization
        total_loss = loss + GAMMA / 2 * weight_decay
        train_loss_acc['loss'] = total_loss.item()
        train_loss_acc['acc'] = train_acc
        optimizer.zero_grad()
        total_loss.backward()
        return total_loss
        
    loss = optimizer.step(closure)
    # check performance on test data
    with torch.no_grad():
        x_test, y_test = next(iter(test_loader))
        x_test, y_test = x_test.to(DEVICE), y_test.to(DEVICE)
        logits = model(x_test)
        test_acc = calculate_accuracy(logits, y_test)
        if test_acc == 1.0:
            end_time = time.time()
            total_func_calls = optimizer.state[next(iter(optimizer.state))]['func_evals']
        test_loss = nn.functional.cross_entropy(logits, y_test)

    train_losses.append(train_loss_acc['loss'])
    test_losses.append(train_loss_acc['acc'])
    train_accs.append(train_acc)
    test_accs.append(test_acc)
    if epoch % 1 == 0:
        s = f"Epoch {epoch}: Loss -- Train: {loss.item():.8f} Test: {test_loss.item():.4f}    Accuracy -- Train: {train_acc:.4f} Test: {test_acc:.4f}"
        log_file.write(s + '\n')
        if epoch % 1 == 0:
            print(s)

s = f'Total time: {end_time - initial_time}s    Total func calls: {total_func_calls}'
log_file.write(s + '\n')
print(s)
log_file.close()

Epoch 0: Loss -- Train: 12.71932406 Test: 7.4058    Accuracy -- Train: 1.0000 Test: 0.0000
Epoch 1: Loss -- Train: 9.88711716 Test: 10.0249    Accuracy -- Train: 1.0000 Test: 0.0008
Epoch 2: Loss -- Train: 7.22791570 Test: 12.8357    Accuracy -- Train: 1.0000 Test: 0.0003
Epoch 3: Loss -- Train: 4.53244905 Test: 16.4119    Accuracy -- Train: 1.0000 Test: 0.0008
Epoch 4: Loss -- Train: 3.19096820 Test: 14.0629    Accuracy -- Train: 1.0000 Test: 0.0008
Epoch 5: Loss -- Train: 2.35975189 Test: 10.9443    Accuracy -- Train: 1.0000 Test: 0.0026
Epoch 6: Loss -- Train: 1.78468566 Test: 10.3102    Accuracy -- Train: 1.0000 Test: 0.0042
Epoch 7: Loss -- Train: 1.49533878 Test: 10.3404    Accuracy -- Train: 1.0000 Test: 0.0102
Epoch 8: Loss -- Train: 1.33042254 Test: 9.9240    Accuracy -- Train: 1.0000 Test: 0.0149
Epoch 9: Loss -- Train: 1.21663867 Test: 9.6230    Accuracy -- Train: 1.0000 Test: 0.0214
Epoch 10: Loss -- Train: 1.13561190 Test: 9.3134    Accuracy -- Train: 1.0000 Test: 0.0266
E